In [1]:
# ============================================================================
# APOLLO OPTIMIZER – CNN BENCHMARK (CIFAR-100 + ResNet-18)
# Kaggle-ready FINAL – OFFLINE MODE (shadowhexer/cifar-100)
# All reviewer/EBM comments addressed inline.
#
# QWEN FIXES APPLIED: 
# 1. Fixed Fatal Statistical Paradox (d_z vs p-value) using Paired T-Test.
# 2. Cleaned Metrics to strictly standard ones (Acc, F1, AUC, MCC).
# 3. Kaggle Smart Checkpointing added to bypass 12-hour disconnects.
# 4. Added Pure_AdamW/Pure_Lion baselines for complete ablation.
# 5. Fixed standard Lion update formula inside Apollo.
# 6. CRITICAL KAGGLE FIX: Set num_workers=0 to eliminate Multiprocessing AssertionError.
# ============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import os, random, time, json, pickle, shutil, warnings, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import Image
from sklearn.metrics import (f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix)
from scipy.stats import ttest_rel, t as t_dist # [R2 #3 SOLUTION] Statistically powerful test for small N
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

# -------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------
plt.style.use('default')
sns.set_style("whitegrid")
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

NUM_EPOCHS = 30          # Sufficient for CIFAR-100 behavior analysis
NUM_RUNS = 3             # [R2 #3 SOLUTION] 3 runs for reliable Mean/STD & Stats
SEEDS = [42, 123, 587]   # Fixed seeds
RESULTS_DIR = "/kaggle/working/results_apollo_cifar100"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "latex_tables"), exist_ok=True)

# Checkpoint File for Kaggle Progress Saving
CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "cnn_checkpoint.json")

EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.001
GRADIENT_CLIP_VALUE = 1.0
BATCH_SIZE = 128

try:
    torch.set_float32_matmul_precision('high')
except:
    pass

print("✅ Environment initialized. Ready for Reproducible CNN Multi-seed Training (CIFAR-100).")

# -------------------------------------------------------------------
# REPRODUCIBILITY
# -------------------------------------------------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    return seed_worker

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

# -------------------------------------------------------------------
# DATASET PREPARATION (OFFLINE KAGGLE)
# [REVIEWER R1 #6 & #7] Source documentation and splits.
# -------------------------------------------------------------------
def prepare_offline_cifar100():
    src_base = '/kaggle/input/datasets/shadowhexer/cifar-100'
    target_base = '/kaggle/working/cifar100_data'
    
    if os.path.exists(os.path.join(src_base, 'cifar-100-python')):
        src_dir = os.path.join(src_base, 'cifar-100-python')
    else:
        src_dir = src_base

    target_sub = os.path.join(target_base, 'cifar-100-python')
    if os.path.exists(os.path.join(target_sub, 'train')):
        return target_base

    os.makedirs(target_base, exist_ok=True)
    if os.path.exists(target_sub):
        shutil.rmtree(target_sub)
    shutil.copytree(src_dir, target_sub)
    return target_base


class CIFAR100Manual(Dataset):
    """Offline-safe CIFAR-100 loader that bypasses torchvision's MD5 checks."""
    def __init__(self, root, train=True, transform=None):
        self.root = root
        self.transform = transform
        base_dir = os.path.join(root, 'cifar-100-python')
        file_name = 'train' if train else 'test'
        with open(os.path.join(base_dir, file_name), 'rb') as f:
            batch = pickle.load(f, encoding='latin1')
        self.data = batch['data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        self.targets = batch['fine_labels']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, target = self.data[idx], int(self.targets[idx])
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, target


def get_cifar100_loaders(batch_size=128, seed=42):
    data_root = prepare_offline_cifar100()
    
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761))
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761))
    ])

    full_train_ds = CIFAR100Manual(root=data_root, train=True, transform=None)
    test_ds       = CIFAR100Manual(root=data_root, train=False, transform=transform_test)

    gen = torch.Generator().manual_seed(seed)
    n_total = len(full_train_ds)
    n_train = int(0.9 * n_total)
    n_val = n_total - n_train
    
    train_subset, val_subset = random_split(full_train_ds, [n_train, n_val], generator=gen)
    train_indices = train_subset.indices
    val_indices   = val_subset.indices

    class SubsetWithTransform(Dataset):
        def __init__(self, dataset, indices, transform=None):
            self.dataset = dataset
            self.indices = indices
            self.transform = transform
        def __len__(self): return len(self.indices)
        def __getitem__(self, idx):
            img, target = self.dataset[self.indices[idx]]
            if self.transform: img = self.transform(img)
            return img, target

    train_ds = SubsetWithTransform(full_train_ds, train_indices, transform_train)
    val_ds   = SubsetWithTransform(full_train_ds, val_indices, transform_test)

    seed_worker_fn = set_seed(seed)
    # [CRITICAL KAGGLE FIX] num_workers=0 completely eliminates the Multiprocessing AssertionError
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True, worker_init_fn=seed_worker_fn)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True, worker_init_fn=seed_worker_fn)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True, worker_init_fn=seed_worker_fn)
    
    return train_loader, val_loader, test_loader

# -------------------------------------------------------------------
# RESNET-18 FOR CIFAR-100
# -------------------------------------------------------------------
class ResNet18_CIFAR100(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.model = models.resnet18(weights=None)
        # Modify for 32x32 image inputs
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.model.maxpool = nn.Identity()
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

# -------------------------------------------------------------------
# OPTIMIZER: APOLLO 
# [EDITOR EBM #2] Conceptual Novelty Implementation via Dynamic Interpolation
# -------------------------------------------------------------------
class Apollo(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999, 0.9), eps=1e-8,
                 weight_decay=1e-4, ablation_mode='standard'):
        if not 0.0 <= lr: raise ValueError(f"Invalid lr: {lr}")
        self.ablation_mode = ablation_mode
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            beta1, beta2, beta3 = group['betas']
            
            if self.ablation_mode == 'no_ema':
                beta3 = 0.0
                
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['update_agreement'] = torch.zeros((), device=p.device)
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                    
                # [QWEN FIX] Standard Lion formulation matching original literature
                lion_c = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                update_lion = torch.sign(lion_c)

                # AdamW formulation
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                m_hat = exp_avg / bc1
                v_hat = exp_avg_sq / bc2
                denom = v_hat.sqrt().add_(group['eps'])
                update_adam = m_hat / denom
                
                # Norm Equalization for geometric linearity consistency
                adam_norm = update_adam.norm(p=2)
                lion_norm = update_lion.norm(p=2)
                if lion_norm > 0:
                    update_lion = update_lion * (adam_norm / lion_norm)
                    
                cos_sim = F.cosine_similarity(update_adam.flatten(), update_lion.flatten(), dim=0, eps=1e-10)
                
                if self.ablation_mode == 'pure_adamw':
                    gamma = 0.0
                elif self.ablation_mode == 'pure_lion':
                    gamma = 1.0
                elif self.ablation_mode == 'no_cosine':
                    gamma = 0.5
                else:
                    state['update_agreement'].mul_(beta3).add_(cos_sim, alpha=1 - beta3)
                    bc3 = 1 - beta3 ** state['step']
                    agreement_hat = state['update_agreement'] / bc3
                    gamma = (agreement_hat + 1.0) / 2.0
                    
                p.add_((1 - gamma) * update_adam + gamma * update_lion, alpha=-group['lr'])
        return loss

# -------------------------------------------------------------------
# BASELINE OPTIMIZERS
# -------------------------------------------------------------------
class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)
    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                grad, state = p.grad, self.state[p]
                if group["weight_decay"] != 0:
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                if "exp_avg" not in state:
                    state["exp_avg"] = torch.zeros_like(p)
                exp_avg = state["exp_avg"]
                beta1, beta2 = group["betas"]
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group["lr"])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)
        return loss

class LAMB(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-6, weight_decay=0):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)
    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                beta1, beta2 = group['betas']
                state['step'] += 1
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                m_hat = exp_avg / (1 - beta1 ** state['step'])
                v_hat = exp_avg_sq / (1 - beta2 ** state['step'])
                update = m_hat / v_hat.sqrt().add_(group['eps'])
                if group['weight_decay'] != 0:
                    update.add_(p, alpha=group['weight_decay'])
                w_norm = p.norm()
                g_norm = update.norm()
                trust_ratio = 1.0 if w_norm == 0 or g_norm == 0 else w_norm / g_norm
                p.add_(update, alpha=-group['lr'] * trust_ratio)
        return loss

class Sophia(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.965, 0.99), rho=0.04, weight_decay=1e-1):
        defaults = dict(lr=lr, betas=betas, rho=rho, weight_decay=weight_decay)
        super().__init__(params, defaults)
    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['hessian'] = torch.zeros_like(p)
                exp_avg, hessian = state['exp_avg'], state['hessian']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if group['weight_decay'] != 0:
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                hessian.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                update = torch.clip(exp_avg / (group['rho'] * hessian + 1e-15), -1, 1)
                p.add_(update, alpha=-group['lr'])
        return loss

# -------------------------------------------------------------------
# TRAINING LOOP & CHECKPOINTING UTILS
# -------------------------------------------------------------------
def load_json(filepath):
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r') as f: return json.load(f)
        except json.JSONDecodeError: pass
    return {}

def save_json(data, filepath):
    with open(filepath, 'w') as f: json.dump(data, f, indent=4)

def train_model(model_fn, opt_factory, train_loader, val_loader, test_loader,
                num_epochs=30, run_name="run", save_cosine_log=False):
    model = model_fn().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = opt_factory(model.parameters())
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_times": []}
    cosine_log = []

    pbar = tqdm(range(num_epochs), desc=f"  📈 {run_name}", leave=True, position=1)
    for epoch in pbar:
        epoch_start = time.time()
        model.train()
        tr_loss, tr_correct, total = 0.0, 0.0, 0

        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad()

            if use_amp:
                with torch.cuda.amp.autocast():
                    out = model(x)
                    loss = criterion(out, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x)
                loss = criterion(out, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                optimizer.step()

            tr_loss += loss.item() * x.size(0)
            tr_correct += out.argmax(1).eq(y).sum().item()
            total += y.size(0)

        epoch_time = time.time() - epoch_start
        history['epoch_times'].append(epoch_time)
        history['train_loss'].append(tr_loss / total)
        history['train_acc'].append(tr_correct / total)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0.0, 0
        gamma_val = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                out = model(x)
                val_loss += criterion(out, y).item() * x.size(0)
                val_correct += out.argmax(1).eq(y).sum().item()
                val_total += y.size(0)

            if optimizer.param_groups and optimizer.param_groups[0]['params']:
                p0 = optimizer.param_groups[0]['params'][0]
                if p0 in optimizer.state and 'update_agreement' in optimizer.state[p0]:
                    gamma_val = optimizer.state[p0]['update_agreement'].item()

        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(val_correct / val_total)
        cosine_log.append(gamma_val)

        pbar.set_postfix({'tr_loss': f"{tr_loss/total:.3f}", 'val_acc': f"{val_correct/val_total:.3f}"})

        if history['val_loss'][-1] < best_val_loss - EARLY_STOPPING_MIN_DELTA:
            best_val_loss = history['val_loss'][-1]
            epochs_no_improve = 0
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            if best_model_state: model.load_state_dict(best_model_state)
            pbar.set_description(f"  ✅ {run_name} (Early Stop)")
            break
        scheduler.step()
    pbar.close()

    # Final test evaluation
    model.eval()
    preds, targets, probs_list = [], [], []
    with torch.no_grad():
        for x, y in tqdm(test_loader, desc=f"  🧪 Testing {run_name}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            probs = F.softmax(out, dim=1).cpu().numpy()
            probs_list.append(probs)
            preds.extend(out.argmax(1).cpu().numpy())
            targets.extend(y.cpu().numpy())

    preds = np.array(preds)
    targets = np.array(targets)
    probs = np.vstack(probs_list)

    # [QWEN FIX] Pruned metrics for layout optimization
    auc_score = roc_auc_score(targets, probs, multi_class='ovr', average='macro')

    metrics = {
        "test_acc": float((preds == targets).mean()),
        "test_f1": float(f1_score(targets, preds, average='macro', zero_division=0)),
        "test_auc": float(auc_score),
        "test_mcc": float(matthews_corrcoef(targets, preds)),
        "final_train_loss": float(history['train_loss'][-1]),
        "avg_epoch_time": float(np.mean(history['epoch_times'])),
        "history": history
    }

    if save_cosine_log and cosine_log:
        with open(os.path.join(RESULTS_DIR, f"cosine_log_{run_name}.csv"), "w") as f:
            w = csv.writer(f)
            w.writerow(["Epoch", "Avg_Update_Agreement"])
            for i, v in enumerate(cosine_log): w.writerow([i+1, v])
            
    return metrics

# -------------------------------------------------------------------
# STATISTICAL ANALYSIS 
# -------------------------------------------------------------------
def calculate_bootstrap_ci(data, n_bootstrap=5000, ci_level=0.95):
    n = len(data)
    if n < 2: return data[0], data[0]
    rng = np.random.default_rng(42)
    boots = [np.mean(rng.choice(data, size=n, replace=True)) for _ in range(n_bootstrap)]
    alpha = (1 - ci_level) / 2
    return float(np.percentile(boots, 100 * alpha)), float(np.percentile(boots, 100 * (1 - alpha)))

def perform_statistical_tests(results, target='test_acc'):
    if 'Apollo' not in results or len(results['Apollo']) < 2:
        return {}
    apollo = np.array([r[target] for r in results['Apollo']])
    sig = {}
    
    for base, runs in results.items():
        if base.startswith('Apollo') or base == 'Apollo' or len(runs) < 2:
            continue
        base_vals = np.array([r[target] for r in runs])
        if len(apollo) != len(base_vals):
            continue
            
        # [QWEN FIX] Standard deviation of paired sample differences to fix d_z calculation paradox
        diffs = apollo - base_vals
        mean_diff = float(np.mean(diffs))
        std_diff = float(np.std(diffs, ddof=1))
        
        if std_diff < 1e-8:
            d_z = 0.0
            p_ttest = 1.0 if abs(mean_diff) < 1e-8 else 0.0
        else:
            d_z = float(mean_diff / std_diff)
            try:
                # [R2 #3 SOLUTION] Paired samples t-test replaces Wilcoxon for low-N power consistency
                _, p_ttest = ttest_rel(apollo, base_vals)
            except:
                p_ttest = 1.0

        sig[base] = {'p_raw_ttest': float(p_ttest), 'diff': mean_diff, 'es': d_z}
        
    if not sig: return {}
    p_vals = [v['p_raw_ttest'] for v in sig.values()]
    _, p_corr, _, _ = multipletests(p_vals, method='fdr_bh')
    
    for i, k in enumerate(sig.keys()):
        sig[k]['p_corr'] = float(p_corr[i])
        sig[k]['sig'] = '***' if p_corr[i]<0.001 else '**' if p_corr[i]<0.01 else '*' if p_corr[i]<0.05 else 'ns'
    return sig

def calculate_variance_statistics(multi_run):
    agg, var, cvs, ci = {}, {}, {}, {}
    metrics_list = ['test_acc', 'test_f1', 'test_auc', 'test_mcc', 'final_train_loss', 'avg_epoch_time']
    for opt, runs in multi_run.items():
        if not runs: continue
        agg[opt], var[opt], cvs[opt], ci[opt] = {}, {}, {}, {}
        for m in metrics_list:
            vals = [r[m] for r in runs]
            mean_val = float(np.mean(vals))
            std_val = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            agg[opt][f'{m}_mean'] = mean_val
            agg[opt][f'{m}_std'] = std_val
            agg[opt][f'{m}_min'] = float(np.min(vals))
            agg[opt][f'{m}_max'] = float(np.max(vals))
            var[opt][f'{m}_var'] = float(np.var(vals, ddof=1)) if len(vals) > 1 else 0.0
            cvs[opt][f'{m}_cv'] = float((std_val / mean_val) * 100) if mean_val > 0 else 0.0
            ci[opt][m] = calculate_bootstrap_ci(vals)
    return agg, var, cvs, ci

# -------------------------------------------------------------------
# LaTeX TABLE GENERATORS
# -------------------------------------------------------------------
def generate_performance_table(agg, ci):
    metrics_config = [
        {'display': 'Acc', 'key': 'test_acc', 'fmt': '.4f'},
        {'display': 'Min/Max Acc', 'key': 'test_acc', 'is_minmax': True, 'fmt': '.4f'},
        {'display': 'F1', 'key': 'test_f1', 'fmt': '.4f'},
        {'display': 'AUC', 'key': 'test_auc', 'fmt': '.4f'},
        {'display': 'MCC', 'key': 'test_mcc', 'fmt': '.4f'},
        {'display': 'Time(s)', 'key': 'avg_epoch_time', 'fmt': '.2f'}
    ]
    lines = [
        "\\begin{table}[htbp]", "\\centering",
        "\\caption{Comprehensive Performance Metrics on CIFAR-100 with ResNet-18 (Mean $\\pm$ SD [95\\% CI])}",
        "\\label{tab:performance_cifar100}", "\\resizebox{\\textwidth}{!}{%",
        "\\begin{tabular}{l" + "c" * len(metrics_config) + "}",
        "\\toprule",
        "{\\bf Optimizer} & " + " & ".join([f"{{\\bf {m['display']}}}" for m in metrics_config]) + " \\\\",
        "\\midrule"
    ]
    order = ['Apollo', 'LAMB', 'Lion', 'AdamW', 'Sophia',
             'Apollo_no_ema', 'Apollo_no_cosine', 'Pure_AdamW', 'Pure_Lion']
    for opt in order:
        if opt not in agg: continue
        row = [opt.replace('_', '\\_')]
        for m in metrics_config:
            if m.get('is_minmax'):
                min_v = agg[opt][f"{m['key']}_min"]
                max_v = agg[opt][f"{m['key']}_max"]
                row.append(f"[{min_v:{m['fmt']}}, {max_v:{m['fmt']}}]")
            else:
                mean = agg[opt][f"{m['key']}_mean"]
                std  = agg[opt][f"{m['key']}_std"]
                c1, c2 = ci[opt][m['key']]
                row.append(f"{mean:{m['fmt']}} $\\pm$ {std:{m['fmt']}} \\; [{c1:{m['fmt']}}, {c2:{m['fmt']}}]")
        lines.append(" & ".join(row) + " \\\\")
    lines.extend(["\\bottomrule", "\\end{tabular}", "}", "\\end{table}\n"])
    return "\n".join(lines)

def generate_statistical_tests_table(sig):
    lines = [
        "\\begin{table}[htbp]", "\\centering",
        "\\caption{Statistical Significance (Paired T-Test + FDR) - ResNet-18 on CIFAR-100}",
        "\\label{tab:stats_cifar100}",
        "\\begin{tabular}{lccccc}", "\\toprule",
        "{\\bf Baseline} & {$\\Delta$ Acc} & {$p_{\\text{ttest}}$} & {$p_{\\text{FDR-corr}}$} & {Effect Size ($d_z$)} & {Sig.} \\\\",
        "\\midrule"
    ]
    for b, r in sig.items():
        diff_str = f"${r['diff']:+.4f}$" if abs(r['diff']) >= 1e-4 else f"${r['diff']:+.1e}$"
        lines.append(f"{b} & {diff_str} & {r['p_raw_ttest']:.4f} & {r['p_corr']:.4f} & {r['es']:.3f} & {r['sig']} \\\\")
    lines.extend(["\\bottomrule", "\\end{tabular}", "\\end{table}\n"])
    return "\n".join(lines)

def generate_variance_cv_table(var, cvs):
    metrics_to_show = [('test_acc', 'Accuracy'), ('test_f1', 'F1-Score'),
                       ('test_auc', 'AUC'), ('final_train_loss', 'Train Loss')]
    lines = [
        "\\begin{table}[htbp]", "\\centering",
        "\\caption{Stability Analysis: Variance ($\\sigma^2$) and Coefficient of Variation (CV \\%) - ResNet-18 on CIFAR-100}",
        "\\label{tab:variance_cifar100}",
        "\\begin{tabular}{l" + "cc" * len(metrics_to_show) + "}",
        "\\toprule",
        "\\multirow{2}{*}{\\bf Optimizer} " + "".join([f"& \\multicolumn{{2}}{{c}}{{\\bf {disp}}}" for _, disp in metrics_to_show]) + "\\\\",
        "\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7} \\cmidrule(lr){8-9}",
        " & " + " & ".join(["$\\sigma^2$ & CV (\\%)"] * len(metrics_to_show)) + " \\\\",
        "\\midrule"
    ]
    order = ['Apollo', 'LAMB', 'Lion', 'AdamW', 'Sophia']
    for opt in order:
        if opt not in var: continue
        row = [opt]
        for key, _ in metrics_to_show:
            v = var[opt][f"{key}_var"]
            c = cvs[opt][f"{key}_cv"]
            if v < 1e-6: row.extend([r"$<10^{-6}$", f"{c:.2f}\\%"])
            else: row.extend([f"{v:.2e}", f"{c:.2f}\\%"])
        lines.append(" & ".join(row) + " \\\\")
    lines.extend(["\\bottomrule", "\\end{tabular}", "\\end{table}\n"])
    return "\n".join(lines)

# -------------------------------------------------------------------
# PLOTTING
# [REVIEWER R1 #11] High resolution image export (300 DPI)
# -------------------------------------------------------------------
def plot_learning_curves(results, dataset_name="CIFAR-100", save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors = plt.cm.tab10.colors
    line_styles = ['-', '--', '-.', ':']

    main_opts = [k for k in results.keys() if 'no_' not in k and 'Pure_' not in k]

    for idx, name in enumerate(main_opts):
        res = results[name]
        if len(res) == 0: continue
        max_e = max((len(r['history']['train_acc']) for r in res), default=0)
        if max_e == 0: continue

        padded_t_acc = np.array([np.pad(r['history']['train_acc'], (0, max_e - len(r['history']['train_acc'])), constant_values=np.nan) for r in res])
        padded_v_acc = np.array([np.pad(r['history']['val_acc'], (0, max_e - len(r['history']['val_acc'])), constant_values=np.nan) for r in res])
        padded_t_loss = np.array([np.pad(r['history']['train_loss'], (0, max_e - len(r['history']['train_loss'])), constant_values=np.nan) for r in res])
        padded_v_loss = np.array([np.pad(r['history']['val_loss'], (0, max_e - len(r['history']['val_loss'])), constant_values=np.nan) for r in res])

        epochs = range(1, max_e + 1)
        c_idx = idx % len(colors)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            mean_t_acc, std_t_acc = np.nanmean(padded_t_acc, axis=0), np.nanstd(padded_t_acc, axis=0)
            mean_v_acc, std_v_acc = np.nanmean(padded_v_acc, axis=0), np.nanstd(padded_v_acc, axis=0)
            mean_t_loss, std_t_loss = np.nanmean(padded_t_loss, axis=0), np.nanstd(padded_t_loss, axis=0)
            mean_v_loss, std_v_loss = np.nanmean(padded_v_loss, axis=0), np.nanstd(padded_v_loss, axis=0)

        ax1.plot(epochs, mean_t_acc, label=f'{name} Train', color=colors[c_idx], linewidth=2, linestyle=line_styles[0])
        ax1.plot(epochs, mean_v_acc, label=f'{name} Val', color=colors[c_idx], linewidth=2, linestyle=line_styles[1], alpha=0.8)
        ax1.fill_between(epochs, mean_t_acc - std_t_acc, mean_t_acc + std_t_acc, color=colors[c_idx], alpha=0.1)
        ax1.fill_between(epochs, mean_v_acc - std_v_acc, mean_v_acc + std_v_acc, color=colors[c_idx], alpha=0.1)

        ax2.plot(epochs, mean_t_loss, label=f'{name} Train', color=colors[c_idx], linewidth=2, linestyle=line_styles[0])
        ax2.plot(epochs, mean_v_loss, label=f'{name} Val', color=colors[c_idx], linewidth=2, linestyle=line_styles[1], alpha=0.8)
        ax2.fill_between(epochs, mean_t_loss - std_t_loss, mean_t_loss + std_t_loss, color=colors[c_idx], alpha=0.1)
        ax2.fill_between(epochs, mean_v_loss - std_v_loss, mean_v_loss + std_v_loss, color=colors[c_idx], alpha=0.1)

    ax1.set_title(f'ResNet-18 on {dataset_name} - Accuracy Learning Curves')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)

    ax2.set_title(f'ResNet-18 on {dataset_name} - Loss Learning Curves')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)

    guidance_text = (
        f"Shaded areas represent ±1 standard deviation across {NUM_RUNS} independent runs.\n"
        "Solid lines: Training, Dashed lines: Validation\n"
        "Smaller shaded areas indicate more stable and reproducible training (Requested by EBM)."
    )
    plt.figtext(0.5, -0.05, guidance_text, ha='center', fontsize=11, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
    plt.tight_layout(rect=[0, 0, 1, 1])

    save_path = save_path or os.path.join(RESULTS_DIR, "plots", "learning_curves_cifar100.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    return save_path

# -------------------------------------------------------------------
# MAIN EXECUTION
# -------------------------------------------------------------------
if __name__ == "__main__":
    print("=" * 80)
    print("APOLLO EXPERIMENT - CNN BENCHMARK (CIFAR-100 + ResNet-18)")
    print("Kaggle Optimized & Statistically Robust (QWEN Fixes Applied)")
    print("=" * 80)
    print(f"Dataset: CIFAR-100 (100 classes) | OFFLINE MODE")
    print(f"Architecture: ResNet-18 (modified for 32x32)")
    print(f"Number of runs: {NUM_RUNS}")
    print(f"Number of training epochs: {NUM_EPOCHS}")

    ModelClass = ResNet18_CIFAR100

    # Include Pure_AdamW and Pure_Lion for complete ablation coverage
    optimizers = {
        'Apollo': lambda p: Apollo(p, lr=1e-3, weight_decay=1e-4),
        'Apollo_no_ema': lambda p: Apollo(p, lr=1e-3, weight_decay=1e-4, ablation_mode='no_ema'),
        'Apollo_no_cosine': lambda p: Apollo(p, lr=1e-3, weight_decay=1e-4, ablation_mode='no_cosine'),
        'Pure_AdamW': lambda p: Apollo(p, lr=1e-3, weight_decay=1e-4, ablation_mode='pure_adamw'),
        'Pure_Lion': lambda p: Apollo(p, lr=1e-3, weight_decay=1e-4, ablation_mode='pure_lion'),
        'LAMB': lambda p: LAMB(p, lr=1e-3, weight_decay=1e-4),
        'Lion': lambda p: Lion(p, lr=1e-4, weight_decay=1e-4),
        'AdamW': lambda p: optim.AdamW(p, lr=1e-3, weight_decay=1e-4),
        'Sophia': lambda p: Sophia(p, lr=1e-4, weight_decay=1e-1),
    }

    # Load Kaggle Smart Checkpoint
    multi_run_results = load_json(CHECKPOINT_FILE)
    for name in optimizers:
        if name not in multi_run_results:
            multi_run_results[name] = []

    for run_idx, seed in enumerate(SEEDS[:NUM_RUNS]):
        print(f"\n{'='*60}")
        print(f"Run {run_idx+1}/{NUM_RUNS} (seed={seed})")
        print(f"{'='*60}")

        train_loader, val_loader, test_loader = get_cifar100_loaders(batch_size=BATCH_SIZE, seed=seed)

        for opt_name, factory in optimizers.items():
            if len([r for r in multi_run_results[opt_name] if r.get('seed') == seed]) > 0:
                print(f"  [Checkpoint] Skipping {opt_name} seed={seed} (Already completed)")
                continue
                
            print(f"  {opt_name:18s}...", end=" ", flush=True)

            set_seed(seed)
            result = train_model(
                lambda: ModelClass(), factory, train_loader, val_loader, test_loader,
                num_epochs=NUM_EPOCHS, run_name=f"{opt_name}_seed{seed}",
                save_cosine_log=(run_idx == 0)
            )
            result['seed'] = seed

            print(f"✓ ({result['avg_epoch_time']:.1f}s/ep, acc: {result['test_acc']:.4f})")
            multi_run_results[opt_name].append(result)
            save_json(multi_run_results, CHECKPOINT_FILE)

    print(f"\n{'='*80}")
    print("CALCULATING STATISTICS AND GENERATING LATEX TABLES")
    print(f"{'='*80}")

    aggregated, variances, cvs, bootstrap_cis = calculate_variance_statistics(multi_run_results)
    significance_results = perform_statistical_tests(multi_run_results, target='test_acc')

    print("\nGenerating LaTeX tables...")
    performance_table = generate_performance_table(aggregated, bootstrap_cis)
    stats_table = generate_statistical_tests_table(significance_results)
    variance_table = generate_variance_cv_table(variances, cvs)

    print("Plotting learning curves...")
    plot_path = plot_learning_curves(multi_run_results, dataset_name="CIFAR-100")

    tables_dir = os.path.join(RESULTS_DIR, "latex_tables")
    os.makedirs(tables_dir, exist_ok=True)

    with open(os.path.join(tables_dir, "table1_performance_cifar100.tex"), "w") as f:
        f.write(performance_table)
    with open(os.path.join(tables_dir, "table2_statistical_tests_cifar100.tex"), "w") as f:
        f.write(stats_table)
    with open(os.path.join(tables_dir, "table3_variance_cv_cifar100.tex"), "w") as f:
        f.write(variance_table)

    summary_path = os.path.join(RESULTS_DIR, "experiment_summary_cifar100.txt")
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("=" * 100 + "\nAPOLLO EXPERIMENT SUMMARY (CIFAR-100 & ResNet-18)\n" + "=" * 100 + "\n\n")
        f.write("SETTINGS:\n" + "-" * 50 + "\n")
        f.write(f"  Runs: {NUM_RUNS}\n  Epochs: {NUM_EPOCHS}\n  Batch size: {BATCH_SIZE}\n  Seeds: {SEEDS[:NUM_RUNS]}\n\n")

        f.write("PERFORMANCE RANKING (Test Accuracy):\n" + "-" * 50 + "\n")
        valid_opts = [(opt, stats) for opt, stats in aggregated.items()]
        ranking = sorted(valid_opts, key=lambda x: x[1]['test_acc_mean'], reverse=True)
        for rank, (opt, stats) in enumerate(ranking, 1):
            mean, std = stats['test_acc_mean'], stats['test_acc_std']
            ci_l, ci_u = bootstrap_cis[opt]['test_acc']
            f.write(f"{rank}. {opt:18s}: {mean:.4f} ± {std:.4f} (CI: [{ci_l:.4f}, {ci_u:.4f}])\n")

        f.write("\nSTATISTICAL SIGNIFICANCE (Apollo vs. Baselines):\n" + "-" * 50 + "\n")
        for baseline, results in significance_results.items():
            f.write(f"  vs {baseline:15s}: p_FDR = {results.get('p_corr', 1.0):.4f} {results.get('sig', '')}, Δ = {results.get('diff', 0):.4f}\n")

    print(f"\n{'='*120}\nEXPERIMENT COMPLETED SUCCESSFULLY\n{'='*120}")
    print(f"\nFiles saved in '{RESULTS_DIR}':")
    print("-" * 80)
    print("✓ latex_tables/table1_performance_cifar100.tex     - Table 1: Standard Metrics")
    print("✓ latex_tables/table2_statistical_tests_cifar100.tex - Table 2: Paired T-Test p-values")
    print("✓ latex_tables/table3_variance_cv_cifar100.tex       - Table 3: Stability/Variance")
    print("✓ plots/learning_curves_cifar100.png                - Learning curves")
    print("✓ experiment_summary_cifar100.txt                  - Text summary")

✅ Environment initialized. Ready for Reproducible CNN Multi-seed Training (CIFAR-100).
🚀 Device: cuda
APOLLO EXPERIMENT - CNN BENCHMARK (CIFAR-100 + ResNet-18)
Kaggle Optimized & Statistically Robust (QWEN Fixes Applied)
Dataset: CIFAR-100 (100 classes) | OFFLINE MODE
Architecture: ResNet-18 (modified for 32x32)
Number of runs: 3
Number of training epochs: 30

Run 1/3 (seed=42)
  Apollo            ... 

  📈 Apollo_seed42:   0%|          | 0/30 [00:00<?, ?it/s]

  🧪 Testing Apollo_seed42:   0%|          | 0/79 [00:00<?, ?it/s]

✓ (41.6s/ep, acc: 0.6833)
  Apollo_no_ema     ... 

  📈 Apollo_no_ema_seed42:   0%|          | 0/30 [00:00<?, ?it/s]

...

  🧪 Testing AdamW_seed587:   0%|          | 0/79 [00:00<?, ?it/s]

✓ (29.6s/ep, acc: 0.6618)
  Sophia            ... 

  📈 Sophia_seed587:   0%|          | 0/30 [00:00<?, ?it/s]

  🧪 Testing Sophia_seed587:   0%|          | 0/79 [00:00<?, ?it/s]

✓ (30.5s/ep, acc: 0.6635)

CALCULATING STATISTICS AND GENERATING LATEX TABLES

Generating LaTeX tables...
Plotting learning curves...

EXPERIMENT COMPLETED SUCCESSFULLY

Files saved in '/kaggle/working/results_apollo_cifar100':
--------------------------------------------------------------------------------
✓ latex_tables/table1_performance_cifar100.tex     - Table 1: Standard Metrics
✓ latex_tables/table2_statistical_tests_cifar100.tex - Table 2: Paired T-Test p-values
✓ latex_tables/table3_variance_cv_cifar100.tex       - Table 3: Stability/Variance
✓ plots/learning_curves_cifar100.png                - Learning curves
✓ experiment_summary_cifar100.txt                  - Text summary
